0.引用库

In [1]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd
import shutil

1.这一段是自动下载

In [1]:
import json
import re
import shutil
import tarfile
import time
from pathlib import Path

import astropy.units as u
import drms
import pandas as pd
import requests
from astropy.io import fits
from astropy.time import Time

# ==================== 参数 ====================
ROOT = Path(r"E:\Learning\PHD3rd\sunspotscar\data\to_be_exam")
EVENTS = ["2015-03-12 12:09:00", "2015-05-05 17:12:00"]
EMAIL = "652024260007@smail.nju.edu.cn"
VECTOR_EMAIL = "19374325@buaa.edu.cn"

MAX_RETRIES = 3
RETRY_SLEEP = 60
EXPORT_TIMEOUT = 7200  # 每次最多等待服务器打包两小时
KEEP_TAR = False       # 解包成功后删除 TAR，节省空间；FITS 保留

GROUPS = [
    ("131", "aia.lev1_euv_12s", ["image"], "36s", 131),
    ("211", "aia.lev1_euv_12s", ["image"], "36s", 211),
    ("304", "aia.lev1_euv_12s", ["image"], "36s", 304),
    ("hmi.Ic_45s", "hmi.Ic_45s", ["continuum"], None, None),
    ("hmi.B_720s", "hmi.B_720s", ["field", "inclination", "azimuth", "disambig"], None, None),
]

ROOT.mkdir(parents=True, exist_ok=True)
REPORT = ROOT / "下载结果.xlsx"
summary_rows, failed_rows = [], []
failed_columns = ["事件UTC", "数据类别", "记录时间TAI", "记录时间UTC", "分量", "原因", "查询语句"]


# 这些操作在十组数据中重复使用，因此集中定义
def retry_query(func, *args, **kwargs):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return func(*args, **kwargs)
        except Exception as exc:
            if attempt == MAX_RETRIES:
                raise
            print(f"  查询失败 {attempt}/{MAX_RETRIES}：{exc}")
            time.sleep(RETRY_SLEEP * attempt)


def time_key(value):
    value = value.decode() if isinstance(value, bytes) else str(value)
    stamp = pd.Timestamp(drms.to_datetime(value))
    if pd.isna(stamp):
        raise ValueError(f"无法解析 T_REC：{value}")
    return stamp.round("ms").isoformat()


def scan_fits(folder, segments):
    """通过 FITS 中的 T_REC 和文件名中的分量识别已有记录。"""
    found = {}
    for path in folder.glob("*.fits"):
        try:
            if path.stat().st_size == 0:
                continue
            if len(segments) == 1:
                segment = segments[0]
            else:
                segment = next((s for s in segments if f".{s}." in path.name.lower()), None)
            if segment is None:
                continue
            with fits.open(path, memmap=True) as hdus:
                hdus.verify("exception")
                trec = next((h.header["T_REC"] for h in hdus if "T_REC" in h.header), None)
            if trec is not None:
                found[(time_key(trec), segment)] = path
        except Exception as exc:
            print(f"  文件暂不计入已完成：{path.name}，{exc}")
    return found


def download_tar(url, target):
    """下载到 .part；支持 HTTP Range 时续传，不支持时重新下载。"""
    part = target.with_name(target.name + ".part")
    offset = part.stat().st_size if part.exists() else 0
    headers = {"Accept-Encoding": "identity"}
    if offset:
        headers["Range"] = f"bytes={offset}-"

    with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as response:
        content_range = response.headers.get("Content-Range", "")
        if response.status_code == 416:
            match = re.fullmatch(r"bytes \*/(\d+)", content_range)
            if match and offset == int(match.group(1)):
                part.replace(target)
                return
            part.unlink(missing_ok=True)
            raise IOError("续传位置无效，下一次将从头下载")

        response.raise_for_status()
        if response.status_code == 206:
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
            if not match or int(match.group(1)) != offset:
                raise IOError("服务器返回的续传范围不符")
            total = int(match.group(3))
            mode = "ab"
        else:
            offset = 0
            mode = "wb"
            length = response.headers.get("Content-Length")
            total = int(length) if length is not None else None

        last_print = time.monotonic()
        with part.open(mode) as f:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                offset += len(chunk)
                if time.monotonic() - last_print >= 30:
                    size_text = f"{offset / 1024**3:.2f} GB"
                    if total:
                        size_text += f" / {total / 1024**3:.2f} GB"
                    print(f"  已下载：{size_text}", flush=True)
                    last_print = time.monotonic()

        if total is not None and part.stat().st_size != total:
            raise IOError("下载长度与服务器声明不符，保留 .part 等待续传")
    part.replace(target)


def unpack_tar(target, folder):
    """只解出普通 FITS 文件，去掉包内目录层级。"""
    with tarfile.open(target, "r:*") as archive:
        members = [m for m in archive.getmembers() if m.isfile() and m.name.lower().endswith(".fits")]
        names = [Path(m.name).name for m in members]
        if not members or len(names) != len(set(names)):
            raise ValueError("TAR 内没有 FITS，或存在重名 FITS")
        for member, name in zip(members, names):
            destination = folder / name
            temp = destination.with_name(destination.name + ".part")
            with archive.extractfile(member) as source, temp.open("wb") as output:
                shutil.copyfileobj(source, output, length=4 * 1024 * 1024)
            if temp.stat().st_size != member.size:
                raise IOError(f"解包大小不符：{name}")
            temp.replace(destination)


from openpyxl import Workbook
from openpyxl.utils import get_column_letter

def write_report():
    book = Workbook()
    summary_sheet = book.active
    summary_sheet.title = "各组汇总"
    failed_sheet = book.create_sheet("未成功下载")
    summary_columns = list(summary_rows[0]) if summary_rows else ["事件UTC", "数据类别", "状态"]

    for sheet, rows, columns in [(summary_sheet, summary_rows, summary_columns), (failed_sheet, failed_rows, failed_columns)]:
        sheet.append(columns)
        for row in rows:
            sheet.append([row.get(col) for col in columns])
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for i, column in enumerate(sheet.columns, start=1):
            width = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column)
            sheet.column_dimensions[get_column_letter(i)].width = min(70, max(14, width + 2))

    temp_report = ROOT / "下载结果.tmp.xlsx"
    book.save(temp_report)
    try:
        temp_report.replace(REPORT)
    except PermissionError:
        alternate = ROOT / f"下载结果_{time.strftime('%Y%m%d_%H%M%S')}.xlsx"
        temp_report.replace(alternate)
        print(f"原 Excel 被占用，本次报告保存为：{alternate}")


# ==================== 开始处理 ====================
for event_text in EVENTS:
    event_time = Time(event_text, scale="utc")
    event_name = event_time.utc.strftime("%Y%m%d_%H%M%S_UTC")
    start = (event_time.tai - 30 * u.minute).strftime("%Y.%m.%d_%H:%M:%S_TAI")
    end = (event_time.tai + 90 * u.minute).strftime("%Y.%m.%d_%H:%M:%S_TAI")

    for group, series, segments, cadence, wavelength in GROUPS:
        folder = ROOT / event_name / group
        folder.mkdir(parents=True, exist_ok=True)
        time_filter = f"{start}-{end}" + (f"@{cadence}" if cadence else "")
        record_query = f"{series}[{time_filter}]" + (f"[{wavelength}]" if wavelength else "")
        export_query = record_query + "{" + ",".join(segments) + "}"
        email = VECTOR_EMAIL if group == "hmi.B_720s" else EMAIL
        state_file = folder / "_jsoc_request.json"
        expected, found = {}, {}
        records = None
        last_error = ""

        print(f"\n{'=' * 65}\n事件：{event_name}，数据：{group}", flush=True)
        print("查询：", export_query)

        try:
            client = retry_query(drms.Client, email=email)
            records = retry_query(client.query, record_query, key="T_REC")
            if records is None or records.empty:
                raise RuntimeError("JSOC 在此查询范围内未返回记录")

            for trec in records["T_REC"]:
                for segment in segments:
                    expected[(time_key(trec), segment)] = str(trec)

            found = scan_fits(folder, segments)
            print(f"服务器记录 {len(records)} 条，预期 FITS {len(expected)} 个")

            for attempt in range(1, MAX_RETRIES + 1):
                missing = set(expected) - set(found)
                if not missing:
                    break

                req = None
                tar_path = None
                print(f"  第 {attempt}/{MAX_RETRIES} 次尝试，目前缺少 {len(missing)} 个 FITS", flush=True)

                try:
                    if state_file.exists():
                        state = json.loads(state_file.read_text(encoding="utf-8"))
                        if state["query"] != export_query or state["email"] != email:
                            raise ValueError(f"请求记录与当前参数不一致，请检查 {state_file}")
                        req = client.export_from_id(state["request_id"])
                    else:
                        req = client.export(export_query, method="url-tar", protocol="fits", email=email)
                        if req.status not in (0, 1, 2, 6):
                            req.wait()  # 显示 status=7 等服务器错误，进入重试
                        if not req.id:
                            raise RuntimeError("服务器未返回请求编号")
                        state = {"query": export_query, "email": email, "request_id": req.id}
                        state_file.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

                    print("  请求编号：", req.id, flush=True)
                    if not req.wait(timeout=EXPORT_TIMEOUT, sleep=30, retries_notfound=10):
                        raise TimeoutError("服务器打包等待超时；请求已保存，下次继续查询")

                    urls = req.urls
                    if not req.tarfile or len(urls) != 1:
                        raise RuntimeError("服务器未返回单个 TAR 下载地址")

                    tar_path = folder / Path(str(urls.iloc[0]["filename"])).name
                    if not tar_path.exists():
                        download_tar(str(urls.iloc[0]["url"]), tar_path)

                    unpack_tar(tar_path, folder)
                    if not KEEP_TAR:
                        tar_path.unlink(missing_ok=True)

                    found = scan_fits(folder, segments)
                    missing = set(expected) - set(found)
                    if not missing:
                        last_error = ""
                        break

                    last_error = f"导出、解包后仍缺少 {len(missing)} 个 FITS"
                    state_file.unlink(missing_ok=True)  # 下一次重新申请打包
                    if tar_path.exists():
                        tar_path.unlink()

                except Exception as exc:
                    last_error = f"{type(exc).__name__}: {exc}"
                    print("  本次失败：", last_error, flush=True)

                    # 失效或明确失败的请求，下次重新提交；排队中的请求继续保留
                    if req is not None and req.status in (-1, 3, 4, 5):
                        state_file.unlink(missing_ok=True)
                    if isinstance(exc, (tarfile.TarError, EOFError)) and tar_path is not None:
                        tar_path.unlink(missing_ok=True)

                if attempt < MAX_RETRIES:
                    print(f"  {RETRY_SLEEP * attempt} 秒后重试", flush=True)
                    time.sleep(RETRY_SLEEP * attempt)

            found = scan_fits(folder, segments)

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            print("  本组处理失败：", last_error, flush=True)

        # 查询成功时，逐条列出缺少的记录及分量
        if expected:
            missing = sorted(set(expected) - set(found))
            for key, segment in missing:
                utc_text = Time(key, format="isot", scale="tai").utc.isot
                failed_rows.append({"事件UTC": event_text, "数据类别": group, "记录时间TAI": expected[(key, segment)], "记录时间UTC": utc_text, "分量": segment, "原因": last_error or "本地没有匹配的有效 FITS", "查询语句": export_query})
            completed = len(expected) - len(missing)
            status = "完成" if not missing else "存在缺失"
        else:
            missing = []
            completed = 0
            status = "查询失败或无记录"
            failed_rows.append({"事件UTC": event_text, "数据类别": group, "记录时间TAI": "", "记录时间UTC": "", "分量": ",".join(segments), "原因": last_error or "无法确定服务器记录清单", "查询语句": export_query})

        summary_rows.append({"事件UTC": event_text, "数据类别": group, "服务器记录数": len(records) if records is not None else None, "预期FITS数": len(expected) if expected else None, "完成FITS数": completed, "缺失FITS数": len(missing) if expected else None, "状态": status, "错误": last_error, "保存目录": str(folder)})
        write_report()
        print(f"  结果：{status}，已完成 {completed} 个 FITS", flush=True)
        time.sleep(10)

print(f"\n全部处理结束。Excel 报告：{REPORT}")
display(pd.DataFrame(summary_rows))


事件：20150312_120900_UTC，数据：131
查询： aia.lev1_euv_12s[2015.03.12_11:39:35_TAI-2015.03.12_13:39:35_TAI@36s][131]{image}
服务器记录 201 条，预期 FITS 201 个
  结果：完成，已完成 201 个 FITS

事件：20150312_120900_UTC，数据：211
查询： aia.lev1_euv_12s[2015.03.12_11:39:35_TAI-2015.03.12_13:39:35_TAI@36s][211]{image}
服务器记录 201 条，预期 FITS 201 个
  第 1/3 次尝试，目前缺少 201 个 FITS
  请求编号： JSOC_20261111_000197


2026-11-11 08:45:17 - drms - INFO: Export request pending. [id=JSOC_20261111_000197, status=2]
2026-11-11 08:45:17 - drms - INFO: Waiting for 30 seconds...
2026-11-11 08:45:48 - drms - INFO: Export request pending. [id=JSOC_20261111_000197, status=1]
2026-11-11 08:45:48 - drms - INFO: Waiting for 30 seconds...
2026-11-11 08:46:20 - drms - INFO: Export request pending. [id=JSOC_20261111_000197, status=1]
2026-11-11 08:46:20 - drms - INFO: Waiting for 30 seconds...
2026-11-11 08:46:51 - drms - INFO: Export request pending. [id=JSOC_20261111_000197, status=1]
2026-11-11 08:46:51 - drms - INFO: Waiting for 30 seconds...
2026-11-11 08:47:23 - drms - INFO: Export request finished. [id=JSOC_20261111_000197, status=0]


  已下载：0.03 GB / 2.00 GB
  已下载：0.08 GB / 2.00 GB
  已下载：0.12 GB / 2.00 GB
  已下载：0.23 GB / 2.00 GB
  已下载：0.30 GB / 2.00 GB
  已下载：0.36 GB / 2.00 GB
  已下载：0.46 GB / 2.00 GB
  已下载：0.60 GB / 2.00 GB
  已下载：0.66 GB / 2.00 GB
  已下载：0.75 GB / 2.00 GB
  已下载：0.87 GB / 2.00 GB
  已下载：0.94 GB / 2.00 GB
  已下载：1.04 GB / 2.00 GB
  已下载：1.15 GB / 2.00 GB
  本次失败： ConnectionError: HTTPSConnectionPool(host='jsoc1.stanford.edu', port=443): Read timed out.
  60 秒后重试
  第 2/3 次尝试，目前缺少 201 个 FITS
  请求编号： JSOC_20261111_000197
  已下载：1.26 GB / 2.00 GB
  已下载：1.41 GB / 2.00 GB
  已下载：1.57 GB / 2.00 GB
  已下载：1.66 GB / 2.00 GB
  已下载：1.74 GB / 2.00 GB
  已下载：1.88 GB / 2.00 GB
  已下载：1.99 GB / 2.00 GB
  结果：完成，已完成 201 个 FITS

事件：20150312_120900_UTC，数据：304
查询： aia.lev1_euv_12s[2015.03.12_11:39:35_TAI-2015.03.12_13:39:35_TAI@36s][304]{image}
服务器记录 201 条，预期 FITS 201 个
  第 1/3 次尝试，目前缺少 201 个 FITS
  请求编号： JSOC_20261111_000288


2026-11-11 09:03:06 - drms - INFO: Export request pending. [id=JSOC_20261111_000288, status=2]
2026-11-11 09:03:06 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:03:37 - drms - INFO: Export request pending. [id=JSOC_20261111_000288, status=1]
2026-11-11 09:03:37 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:04:09 - drms - INFO: Export request pending. [id=JSOC_20261111_000288, status=1]
2026-11-11 09:04:09 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:04:40 - drms - INFO: Export request finished. [id=JSOC_20261111_000288, status=0]


  已下载：0.07 GB / 1.52 GB
  已下载：0.12 GB / 1.52 GB
  已下载：0.21 GB / 1.52 GB
  已下载：0.32 GB / 1.52 GB
  已下载：0.45 GB / 1.52 GB
  已下载：0.54 GB / 1.52 GB
  已下载：0.58 GB / 1.52 GB
  已下载：0.73 GB / 1.52 GB
  已下载：0.83 GB / 1.52 GB
  已下载：0.91 GB / 1.52 GB
  已下载：1.00 GB / 1.52 GB
  已下载：1.08 GB / 1.52 GB
  已下载：1.13 GB / 1.52 GB
  已下载：1.18 GB / 1.52 GB
  已下载：1.22 GB / 1.52 GB
  已下载：1.28 GB / 1.52 GB
  已下载：1.38 GB / 1.52 GB
  已下载：1.52 GB / 1.52 GB
  结果：完成，已完成 201 个 FITS

事件：20150312_120900_UTC，数据：hmi.Ic_45s
查询： hmi.Ic_45s[2015.03.12_11:39:35_TAI-2015.03.12_13:39:35_TAI]{continuum}
服务器记录 161 条，预期 FITS 161 个
  第 1/3 次尝试，目前缺少 161 个 FITS
  请求编号： JSOC_20261111_000332


2026-11-11 09:14:30 - drms - INFO: Export request pending. [id=JSOC_20261111_000332, status=2]
2026-11-11 09:14:30 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:15:01 - drms - INFO: Export request pending. [id=JSOC_20261111_000332, status=1]
2026-11-11 09:15:01 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:15:33 - drms - INFO: Export request pending. [id=JSOC_20261111_000332, status=1]
2026-11-11 09:15:33 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:16:05 - drms - INFO: Export request pending. [id=JSOC_20261111_000332, status=1]
2026-11-11 09:16:05 - drms - INFO: Waiting for 30 seconds...
2026-11-11 09:16:37 - drms - INFO: Export request finished. [id=JSOC_20261111_000332, status=0]


... (10 outputs left out)

  已下载：0.00 GB / 2.23 GB
  已下载：0.01 GB / 2.23 GB
  已下载：0.01 GB / 2.23 GB
  本次失败： ChunkedEncodingError: ('Connection broken: IncompleteRead(12811958 bytes read, 2386481482 more expected)', IncompleteRead(12811958 bytes read, 2386481482 more expected))
  60 秒后重试
  第 2/3 次尝试，目前缺少 161 个 FITS
  请求编号： JSOC_20261111_002066
  已下载：0.02 GB / 2.23 GB
  本次失败： ChunkedEncodingError: ('Connection broken: IncompleteRead(7536266 bytes read, 2379174262 more expected)', IncompleteRead(7536266 bytes read, 2379174262 more expected))
  120 秒后重试
  第 3/3 次尝试，目前缺少 161 个 FITS
  请求编号： JSOC_20261111_002066
  已下载：0.02 GB / 2.23 GB
  本次失败： ChunkedEncodingError: ('Connection broken: IncompleteRead(7896714 bytes read, 2374619510 more expected)', IncompleteRead(7896714 bytes read, 2374619510 more expected))
  结果：存在缺失，已完成 0 个 FITS

事件：20150505_171200_UTC，数据：hmi.B_720s
查询： hmi.B_720s[2015.05.05_16:42:35_TAI-2015.05.05_18:42:35_TAI]{field,inclination,azimuth,disambig}
服务器记录 11 条，预期 FITS 44 个
  第 1/3 次尝试，目前缺少 44 个 FITS
  请

2026-11-11 15:47:54 - drms - INFO: Export request pending. [id=JSOC_20261111_002269, status=2]
2026-11-11 15:47:54 - drms - INFO: Waiting for 30 seconds...
2026-11-11 15:48:25 - drms - INFO: Export request pending. [id=JSOC_20261111_002269, status=1]
2026-11-11 15:48:25 - drms - INFO: Waiting for 30 seconds...
2026-11-11 15:48:57 - drms - INFO: Export request pending. [id=JSOC_20261111_002269, status=1]
2026-11-11 15:48:57 - drms - INFO: Waiting for 30 seconds...
2026-11-11 15:49:28 - drms - INFO: Export request finished. [id=JSOC_20261111_002269, status=0]


  已下载：0.00 GB / 0.54 GB
  已下载：0.01 GB / 0.54 GB
  已下载：0.01 GB / 0.54 GB
  本次失败： ChunkedEncodingError: ('Connection broken: IncompleteRead(15892151 bytes read, 564449609 more expected)', IncompleteRead(15892151 bytes read, 564449609 more expected))
  60 秒后重试
  第 2/3 次尝试，目前缺少 44 个 FITS
  请求编号： JSOC_20261111_002269
  已下载：0.02 GB / 0.54 GB
  本次失败： ChunkedEncodingError: ('Connection broken: IncompleteRead(7224973 bytes read, 560533875 more expected)', IncompleteRead(7224973 bytes read, 560533875 more expected))
  120 秒后重试
  第 3/3 次尝试，目前缺少 44 个 FITS
  请求编号： JSOC_20261111_002269


KeyboardInterrupt: 

2.这段代码是把一块下好的AIA不同波段分出来

In [15]:
#这段代码是把一块下好的AIA不同波段分出来
list1=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\131,304\*.fits')
list1.sort()
# print(list1[0][-19:-16])
for i in range(len(list1)):
    if list1[i][-19:-16]=='131':
        shutil.copy(list1[i],r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\131')

    if list1[i][-19:-16]=='304':
        shutil.copy(list1[i],r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\304')

3.实现从720s磁场的系列直接生成Br和倾角的系列变化

In [15]:
from pathlib import Path
import gc
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs.utils import pixel_to_skycoord
import sunpy.map
from sunpy.coordinates import frames


# ==================== 1. 路径和参数 ====================

root = Path(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam')
input_dir = root / "hmi.B_720s"

output_dirs = {
    "Bxyz": root / "hmi_bxyz",
    "Br": root / "hmi_Br",
    "Btheta": root / "hmi_Btheta",
    "Bphi": root / "hmi_Bphi",
    "inclination": root / "inclination",
}

block_rows = 256       # 每次处理 256 行，减少全日面坐标转换的内存占用
limit = 10000.0        # 超过此范围的磁场分量标为 NaN，单位 G
overwrite = False     # 如需覆盖已经生成的文件，改为 True


# ==================== 2. 翻转后同步更新坐标头 ====================
def rotated_meta(base):
    """对应 np.rot90(data, 2)，只翻转像素，不进行插值。"""
    ny, nx = base.data.shape
    meta = base.meta.copy()
    meta["crpix1"] = nx + 1 - base.meta["crpix1"]
    meta["crpix2"] = ny + 1 - base.meta["crpix2"]

    for key in ["crota1", "crota2", "blank", "bscale", "bzero", "datamin", "datamax", "checksum", "datasum"]:
        meta.pop(key, None)

    meta["pc1_1"], meta["pc1_2"], meta["pc2_1"], meta["pc2_2"] = (-base.wcs.wcs.pc).ravel()
    meta["bunit"] = "G"
    meta["rot180"] = True
    return meta


# ==================== 3. 原始矢量磁图 → Bxyz ====================
def make_bxyz(paths, stamp):
    base = sunpy.map.Map(paths["field"])
    B = base.data.astype(np.float32)
    inc = fits.getdata(paths["inclination"], memmap=False).astype(np.float32)
    azi = fits.getdata(paths["azimuth"], memmap=False).astype(np.float32)
    dis = fits.getdata(paths["disambig"], memmap=False)

    if not (B.shape == inc.shape == azi.shape == dis.shape):
        raise ValueError(f"{stamp}: 四个原始数组的尺寸不一致")

    # 原始 inclination 是相对视线方向的角度，不是最后要计算的局部倾角。
    bad = (~np.isfinite(B)) | (~np.isfinite(inc)) | (~np.isfinite(azi)) | (~np.isfinite(dis))
    bad |= (B < 0) | (B > limit) | (inc < 0) | (inc > 180)

    # 沿用参考 notebook 的消歧方案和 Bxyz 定义。
    azi[dis > 3] += 180.0
    gamma = np.deg2rad(inc)
    psi = np.deg2rad(azi + 270.0)

    Bx = B * np.sin(gamma) * np.cos(psi)
    By = B * np.sin(gamma) * np.sin(psi)
    Bz = B * np.cos(gamma)

    # 无效像素统一设为 NaN，避免继续生成看似有效的倾角。
    Bx[bad], By[bad], Bz[bad] = np.nan, np.nan, np.nan
    Bx = np.rot90(Bx, 2).copy()
    By = np.rot90(By, 2).copy()
    Bz = np.rot90(Bz, 2).copy()

    meta = rotated_meta(base)
    output_path = output_dirs["Bxyz"] / f"hmi.B_720s.{stamp}.Bxyz.fits"

    # 一个 FITS 保存三个分量，可用 fits.getdata(path, "Bx") 等方式读取。
    hdus = []
    for index, (name, data) in enumerate([("Bx", Bx), ("By", By), ("Bz", Bz)]):
        component_meta = meta.copy()
        component_meta["bcomp"] = name
        component_meta["content"] = f"HMI {name}"
        header = sunpy.map.Map(data, component_meta).fits_header
        header["EXTNAME"] = name
        hdu = fits.PrimaryHDU(data, header) if index == 0 else fits.ImageHDU(data, header, name=name)
        hdus.append(hdu)

    with fits.HDUList(hdus) as hdul:
        hdul.writeto(output_path, overwrite=True)

    return output_path


# ==================== 4. Bxyz → Br、Btheta、Bphi → 倾角 ====================
def make_brthetaphi_inclination(bxyz_path, stamp):
    with fits.open(bxyz_path, memmap=True) as hdul:
        B_map = sunpy.map.Map(hdul["Bx"].data, hdul["Bx"].header)
        ny, nx = B_map.data.shape
        components = {name: np.full((ny, nx), np.nan, dtype=np.float32) for name in ["Br", "Btheta", "Bphi", "inclination"]}

        # 从翻转后的 WCS 获取坐标参数，不再额外给 p 加 π。
        phi0 = B_map.observer_coordinate.lon.to_value(u.rad)
        b = B_map.observer_coordinate.lat.to_value(u.rad)
        pc = B_map.wcs.wcs.pc
        p = np.arctan2(pc[0, 1], pc[0, 0])
        sb, cb, sp, cp = np.sin(b), np.cos(b), np.sin(p), np.cos(p)

        for y0 in range(0, ny, block_rows):
            y1 = min(y0 + block_rows, ny)
            rows = slice(y0, y1)
            Bx, By, Bz = [hdul[name].data[rows].astype(np.float64) for name in ["Bx", "By", "Bz"]]

            # 每个像素对应的太阳经纬度；日面外位置会产生 NaN。
            y, x = np.mgrid[y0:y1, 0:nx]
            coords = pixel_to_skycoord(x, y, B_map.wcs)
            hg = coords.transform_to(frames.HeliographicStonyhurst)
            dphi = hg.lon.to_value(u.rad) - phi0
            lam = hg.lat.to_value(u.rad)
            sl, cl, sd, cd = np.sin(lam), np.cos(lam), np.sin(dphi), np.cos(dphi)

            # 沿用参考 notebook 的变换矩阵。
            k11 = cl * (sb * sp * cd + cp * sd) - sl * cb * sp
            k12 = -cl * (sb * cp * cd - sp * sd) + sl * cb * cp
            k13 = cl * cb * cd + sl * sb
            k21 = sl * (sb * sp * cd + cp * sd) + cl * cb * sp
            k22 = -sl * (sb * cp * cd - sp * sd) - cl * cb * cp
            k23 = sl * cb * cd - cl * sb
            k31 = -sb * sp * sd + cp * cd
            k32 = sb * cp * sd + sp * cd
            k33 = -cb * sd

            # Br 向外为正，Btheta 向南为正，Bphi 向西为正。
            Br = (k11 * Bx + k12 * By + k13 * Bz).astype(np.float32)
            Btheta = (k21 * Bx + k22 * By + k23 * Bz).astype(np.float32)
            Bphi = (k31 * Bx + k32 * By + k33 * Bz).astype(np.float32)

            bad = (~np.isfinite(Bx)) | (~np.isfinite(By)) | (~np.isfinite(Bz))
            for arr in [Br, Btheta, Bphi]:
                bad |= (~np.isfinite(arr)) | (np.abs(arr) > limit)
            for arr in [Br, Btheta, Bphi]:
                arr[bad] = np.nan

            # 相对局部径向轴的倾角：竖直 0°，水平 90°，不区分磁极性。
            Bh = np.hypot(Btheta, Bphi)
            inclination = np.degrees(np.arctan2(Bh, np.abs(Br))).astype(np.float32)
            inclination[(Bh == 0) & (Br == 0)] = np.nan  # 零磁场没有确定方向

            components["Br"][rows] = Br
            components["Btheta"][rows] = Btheta
            components["Bphi"][rows] = Bphi
            components["inclination"][rows] = inclination

        # 各个物理量分别保存，保留与数组匹配的坐标头和原始观测时间。
        for name, data in components.items():
            meta = B_map.meta.copy()
            meta.pop("extname", None)
            meta["bunit"] = "deg" if name == "inclination" else "G"
            meta["bcomp"] = name
            meta["content"] = "Local inclination, 0-90 deg" if name == "inclination" else f"HMI {name}"
            output_path = output_dirs[name] / f"hmi.B_720s.{stamp}.{name}.fits"
            sunpy.map.Map(data, meta).save(output_path, overwrite=True)


# ==================== 5. 检查并处理全部 21 个时刻 ====================

# input_dir：原始数据文件夹。
# field_files：所有 field.fits 文件的路径列表；每个文件对应一个时刻。
# glob() 按文件名模式查找文件，sorted() 按文件名排序，这批文件也就按时间排好了。
field_files = sorted(input_dir.glob("hmi.B_720s.*.field.fits"))
# if len(field_files) != 21:
#     raise ValueError(f"预计有 21 组数据，实际找到 {len(field_files)} 个 field 文件，请检查输入目录")

# records：保存每个时刻的时间标记，以及该时刻所需的四个原始文件路径。
records = []
for field_path in field_files:
    # field_path：当前这个 field.fits 文件的完整路径；field_path.name 只取文件名。
    # 去掉文件名前后固定的部分，得到时间标记，例如 20240618_200000_TAI。
    stamp = field_path.name.removeprefix("hmi.B_720s.").removesuffix(".field.fits")

    # paths：字典，根据同一个时间标记拼出四个文件路径。
    # 例如 paths["field"] 是场强文件路径，paths["azimuth"] 是方位角文件路径。
    paths = {name: input_dir / f"hmi.B_720s.{stamp}.{name}.fits" for name in ["field", "inclination", "azimuth", "disambig"]}

    # 先检查四个文件是否齐全，缺少任何一个就停止并指出文件名。
    for path in paths.values():
        if not path.is_file():
            raise FileNotFoundError(f"缺少文件：{path}")

    # 将当前时刻的时间标记和文件路径放进 records，供后面的处理循环使用。
    records.append((stamp, paths))

# output_dirs：前面定义的输出文件夹字典；values() 取出其中的所有文件夹路径。
# parents=True：上级文件夹不存在时也一起创建。
# exist_ok=True：文件夹已经存在时不报错。
for folder in output_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

# 依次处理每个时刻：
# index 是处理序号，从 1 开始；stamp 是时间标记；paths 是该时刻的四个输入文件路径。
for index, (stamp, paths) in enumerate(records, start=1):
    print(f"[{index}] 正在处理 {stamp}", flush=True)  # flush=True 让进度及时显示

    # 第一步：读取原始矢量磁图，生成并保存 Bxyz。
    # bxyz_path：生成的 Bxyz.fits 文件的完整路径。
    bxyz_path = make_bxyz(paths, stamp)
    gc.collect()  # 尝试回收已经不再使用的对象，减少连续处理时的内存占用

    # 第二步：读取刚保存的 Bxyz，计算并保存 Br、Btheta、Bphi 和局部倾角。
    make_brthetaphi_inclination(bxyz_path, stamp)
    gc.collect()

    print(f"[{index}] 已保存 Bxyz、Br、Btheta、Bphi、inclination", flush=True)

print("全部处理完成！文件名中的时间保留原始 TAI 标记。")

[1] 正在处理 20130411_080000_TAI
[1] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[2] 正在处理 20130411_081200_TAI
[2] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[3] 正在处理 20130411_082400_TAI
[3] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[4] 正在处理 20130411_083600_TAI
[4] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[5] 正在处理 20130411_084800_TAI
[5] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[6] 正在处理 20130411_090000_TAI
[6] 已保存 Bxyz、Br、Btheta、Bphi、inclination
全部处理完成！文件名中的时间保留原始 TAI 标记。


3.1 上面那个是同一个事件生成很多时刻的Br，下面这个是多个事件每个事件只生成一个Br

In [ ]:
from pathlib import Path

import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs.utils import pixel_to_skycoord
import sunpy.map
from sunpy.coordinates import frames

root = Path(r'E:\Learning\PHD3rd\sunspotscar\data\C')
block_rows = 256
limit = 10000.0
overwrite = True
save_bh = True

input_dirs = sorted(folder for folder in root.glob('*/hmi.B_720s') if folder.is_dir())
if not input_dirs:
    raise FileNotFoundError(f'没有找到事件的 hmi.B_720s 文件夹：{root}')

names = ['Br', 'Btheta', 'Bphi', 'inclination_calc']
if save_bh:
    names.append('Bh')

n_done = 0
failed = []

for idx, folder in enumerate(input_dirs, start=1):
    field_files = sorted(folder.glob('*.field.fits'))
    if not field_files:
        failed.append((folder, '没有找到 *.field.fits'))
        print(f'[{idx}/{len(input_dirs)}] 缺少场强文件：{folder}', flush=True)
        continue

    for field_path in field_files:
        prefix = field_path.name[:-len('.field.fits')]
        output_prefix = f'{prefix}.' if len(field_files) > 1 else ''
        output_paths = {name: folder / f'{output_prefix}{name}.fits' for name in names}

        if not overwrite and any(path.exists() for path in output_paths.values()):
            print(f'[{idx}/{len(input_dirs)}] 已有输出，跳过：{folder.parent.name} / {prefix}', flush=True)
            continue

        print(f'[{idx}/{len(input_dirs)}] 正在处理 {folder.parent.name} / {prefix}', flush=True)
        try:
            paths = {seg: folder / f'{prefix}.{seg}.fits' for seg in ('inclination', 'azimuth', 'disambig')}
            for path in paths.values():
                if not path.is_file():
                    raise FileNotFoundError(f'缺少同一时刻的输入：{path.name}')

            base = sunpy.map.Map(field_path)
            ny, nx = base.data.shape

            # [::-1, ::-1] 等价于 np.rot90(..., 2)，只重排像素。
            # CRPIX 按 FITS 的 1-based 编号翻转，PC 整体取负，CDELT 不变。
            meta = base.meta.copy()
            meta['crpix1'], meta['crpix2'] = nx + 1 - meta['crpix1'], ny + 1 - meta['crpix2']
            for key in ('crota1', 'crota2', 'cd1_1', 'cd1_2', 'cd2_1', 'cd2_2', 'blank', 'bscale', 'bzero', 'datamin', 'datamax', 'checksum', 'datasum', 'extname'):
                meta.pop(key, None)
            meta['pc1_1'], meta['pc1_2'], meta['pc2_1'], meta['pc2_2'] = (-base.wcs.wcs.get_pc()).ravel()
            meta['bunit'] = 'G'
            meta['rot180'] = True

            B_map = sunpy.map.Map(base.data[::-1, ::-1], meta)
            B = B_map.data
            inc, azi, dis = (fits.getdata(paths[seg], memmap=False)[::-1, ::-1] for seg in ('inclination', 'azimuth', 'disambig'))
            if not (B.shape == inc.shape == azi.shape == dis.shape):
                raise ValueError('field、inclination、azimuth、disambig 的尺寸不一致')

            # p 取自翻转后的 WCS，已经包含 180° 基底变化。
            wcs = B_map.wcs
            pc = wcs.wcs.get_pc()
            p = np.arctan2(pc[0, 1], pc[0, 0])
            phi0 = B_map.observer_coordinate.lon.to_value(u.rad)
            b = B_map.observer_coordinate.lat.to_value(u.rad)
            sb, cb, sp, cp = np.sin(b), np.cos(b), np.sin(p), np.cos(p)
            outputs = {name: np.empty((ny, nx), dtype=np.float32) for name in names}

            for y0 in range(0, ny, block_rows):
                y1 = min(y0 + block_rows, ny)
                rows = slice(y0, y1)
                B0, inc0, azi0, dis0 = B[rows], inc[rows], azi[rows], dis[rows]

                # 沿用原来的消歧和 azi + 270°；四个输入数组已在同一个翻转网格上。
                bad = ~np.isfinite(B0) | ~np.isfinite(inc0) | ~np.isfinite(azi0) | ~np.isfinite(dis0)
                bad |= (B0 < 0) | (B0 > limit) | (inc0 < 0) | (inc0 > 180)
                gamma = np.deg2rad(np.where(bad, 0.0, inc0))
                psi = np.deg2rad(np.where(bad, 0.0, azi0) + 270.0 + np.where(dis0 > 3, 180.0, 0.0))
                B0 = np.where(bad, np.nan, B0)
                Bt = B0 * np.sin(gamma)
                Bx, By, Bz = Bt * np.cos(psi), Bt * np.sin(psi), B0 * np.cos(gamma)

                # 真实日面经纬度：不手动修改 phi 或 lam。
                y, x = np.mgrid[y0:y1, 0:nx]
                hg = pixel_to_skycoord(x, y, wcs).transform_to(frames.HeliographicStonyhurst)
                dphi = hg.lon.to_value(u.rad) - phi0
                lam = hg.lat.to_value(u.rad)
                sl, cl, sd, cd = np.sin(lam), np.cos(lam), np.sin(dphi), np.cos(dphi)

                k11 = cl * (sb * sp * cd + cp * sd) - sl * cb * sp
                k12 = -cl * (sb * cp * cd - sp * sd) + sl * cb * cp
                k13 = cl * cb * cd + sl * sb
                k21 = sl * (sb * sp * cd + cp * sd) + cl * cb * sp
                k22 = -sl * (sb * cp * cd - sp * sd) - cl * cb * cp
                k23 = sl * cb * cd - cl * sb
                k31 = -sb * sp * sd + cp * cd
                k32 = sb * cp * sd + sp * cd
                k33 = -cb * sd

                Br = k11 * Bx + k12 * By + k13 * Bz
                Btheta = k21 * Bx + k22 * By + k23 * Bz
                Bphi = k31 * Bx + k32 * By + k33 * Bz

                for arr in (Br, Btheta, Bphi):
                    bad |= ~np.isfinite(arr) | (np.abs(arr) > limit)
                for arr in (Br, Btheta, Bphi):
                    arr[bad] = np.nan

                Bh = np.hypot(Btheta, Bphi)
                inclination = np.degrees(np.arctan2(Bh, np.abs(Br)))
                inclination[(Bh == 0) & (Br == 0)] = np.nan

                outputs['Br'][rows] = Br
                outputs['Btheta'][rows] = Btheta
                outputs['Bphi'][rows] = Bphi
                outputs['inclination_calc'][rows] = inclination
                if save_bh:
                    outputs['Bh'][rows] = Bh

            for name, data in outputs.items():
                out_meta = meta.copy()
                out_meta['bunit'] = 'deg' if name == 'inclination_calc' else 'G'
                out_meta['bcomp'] = name
                out_meta['content'] = 'Local inclination, 0-90 deg' if name == 'inclination_calc' else f'HMI {name}'
                sunpy.map.Map(data, out_meta).save(output_paths[name], overwrite=overwrite)

            n_done += 1
            print('  已保存：' + '、'.join(path.name for path in output_paths.values()), flush=True)

        except Exception as exc:
            failed.append((field_path, str(exc)))
            print(f'  失败：{exc}', flush=True)

print(f'处理完成：成功保存 {n_done} 帧，失败或缺少输入 {len(failed)} 项。')
for path, reason in failed:
    print(f'  {path}：{reason}')

4.旋转对齐+裁剪  
4.1有裁剪好了的子图，直接对齐过去

In [3]:
#裁剪目标区域并进行旋转对齐
root_dir=r'E:\Learning\PHD3rd\sunspotscar\data\M\20140416_195400_UTC'
file_time=os.listdir(root_dir)
passbands=['304']


for j in passbands:
    file_name=glob.glob(os.path.join(root_dir,j,'*.fits'))
    os.makedirs(os.path.join(root_dir,j+'sub_map'), exist_ok=True)
    sub_file_path=os.path.join(root_dir,j+'sub_map')
    sub_map0=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\M\20140416_195400_UTC\131sub_map\aia.lev1_euv_12s.2014-04-16T192410Z.131.image_lev1.fits')
    for k in range(len(file_name)):
        print(k)
        

        map1=sunpy.map.Map(file_name[k])
        with propagate_with_solar_surface():
            rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
        for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                    'bunit', 'exptime']:
            if key in map1.meta:
                rot_map1.meta[key] = map1.meta[key]

        rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
        rot_map1.meta.pop('BLANK', None)
        rot_map1.save(os.path.join(sub_file_path, os.path.basename(file_name[k])), overwrite=True)
        #rot_map1.save(os.path.join(sub_file_path, os.path.basename(file_name[k][-38:])), overwrite=True)

    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65


4.2没有裁剪好的子图，要裁剪，后面再旋转对齐（记得改路径和x,y range）

In [9]:
#裁剪目标区域并进行旋转对齐
root_dir=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20150505_171200_UTC'
#file_time=os.listdir(root_dir)
# df=pd.read_excel('../dataset/M级耀斑.xlsx')
# #df.drop(df.index[8:16], inplace=True)
# x_range=df['X, arcsec'].values
# y_range=df['Y, arcsec'].values
width=800
passbands=['131','211','304','hmi.Ic_45s']
i=0
x_range=[-221.1]
y_range=[-262.1]

for j in passbands:
    file_name=glob.glob(os.path.join(root_dir,j,'*.fits'))
    os.makedirs(os.path.join(root_dir,j+'sub_map'), exist_ok=True)
    sub_file_path=os.path.join(root_dir,j+'sub_map')
    
    for k in range(len(file_name)):
        print(k)
        
        if k==0:
            map0=sunpy.map.Map(file_name[k])
            bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
            tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
            sub_map0=map0.submap(bl1,top_right=tr1)
            sub_map0.meta.pop('BLANK', None)
            sub_map0.save(os.path.join(sub_file_path, os.path.basename(file_name[k])),overwrite=True)
        else:
            map1=sunpy.map.Map(file_name[k])
            with propagate_with_solar_surface():
                rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
            for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                        'bunit', 'exptime']:
                if key in map1.meta:
                    rot_map1.meta[key] = map1.meta[key]

            rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
            rot_map1.meta.pop('BLANK', None)
            rot_map1.save(os.path.join(sub_file_path, os.path.basename(file_name[k])), overwrite=True)

    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103


4.5 将HMI<font color=cyan>连续谱</font>旋转对齐到特定的时刻

In [4]:
import glob
import os

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.wcs import WCS
import sunpy.map
from sunpy.coordinates import Helioprojective, propagate_with_solar_surface
from datetime import datetime
rootdir=r'E:\Learning\PHD3rd\sunspotscar\data\M'
df=pd.read_excel('../dataset/M级耀斑.xlsx')
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values

for i in range(66,67):
    try:
        print(i)
        root_dir=os.path.join(rootdir,os.listdir(rootdir)[i])
        passbands=['hmi.Ic_45ssub_map']
        time_str=os.listdir(os.path.join(root_dir,'131sub_map'))[0][-38:-21]
        t_ref = Time(datetime.strptime(time_str, '%Y-%m-%dT%H%M%S'), scale='utc')
        ref_path = sorted(glob.glob(os.path.join(root_dir, passbands[0], '*.fits')))[0]
        ref_map = sunpy.map.Map(ref_path)
        frame_ref = Helioprojective(observer='earth', obstime=t_ref, rsun=ref_map.rsun_meters)
        center_ref = SkyCoord(x_range[i] * u.arcsec, y_range[i] * u.arcsec, frame=frame_ref)

        width=800
        pixel_scale = 0.6  # arcsec/pixel，所有波段共用
        n = round(width / pixel_scale)  # width 沿用你的 800 arcsec
        shape_out = (n, n)
        header_ref = sunpy.map.make_fitswcs_header(shape_out, center_ref, scale=[pixel_scale, pixel_scale] * u.arcsec / u.pix, rotation_matrix=ref_map.rotation_matrix)
        target_wcs = WCS(header_ref)

        for j in passbands:
            sub_file_path = os.path.join(root_dir, j + '_aligned')
            os.makedirs(sub_file_path, exist_ok=True)

            for path in sorted(glob.glob(os.path.join(root_dir, j, '*.fits'))):
                map1 = sunpy.map.Map(path)
                with propagate_with_solar_surface():
                    rot_map1 = map1.reproject_to(target_wcs, shape_out=shape_out, preserve_date_obs=True)

                for key in ('telescop', 'instrume', 'detector', 'wavelnth', 'waveunit', 'bunit', 'pixlunit', 'exptime', 'content'):
                    if key in map1.meta:
                        rot_map1.meta[key] = map1.meta[key]

                rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
                rot_map1.meta.pop('BLANK', None)
                rot_map1.save(os.path.join(sub_file_path, os.path.basename(path)), overwrite=True)
    except:
        print(i,'is wrong')

66


4.3 对hmi continuum的要和自己的对齐，要不然会差.rotate()

In [ ]:
#hmi continuum的要和自己的对齐，要不然会差.rotate()
root_dir=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam'
file_time=os.listdir(root_dir)

passbands=['cont']
for j in passbands:
    file_name=glob.glob(os.path.join(root_dir,j,'*.fits'))
    os.makedirs(os.path.join(root_dir,j+'sub_map'), exist_ok=True)
    sub_file_path=os.path.join(root_dir,j+'sub_map')
    sub_map0=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi.Ic_45ssub_map\s.20130411_062445_TAI.2.continuum.fits')
    for k in range(len(file_name)):
        print(k)
        map1=sunpy.map.Map(file_name[k])
        with propagate_with_solar_surface():
            rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
        for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                    'bunit', 'exptime']:
            if key in map1.meta:
                rot_map1.meta[key] = map1.meta[key]

        rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
        rot_map1.meta.pop('BLANK', None)
        rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)

    

4.4 对Br和inclination的裁剪旋转，他们的时间差别比较大，因此要去和131的最初的时刻对齐

In [16]:
import glob
ref_sub_map=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\131sub_map\aia.lev1_euv_12s.2013-04-11T062510Z.131.image_lev1.fits')

file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\hmi_Br\*.fits')
save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\hmi_Br_submap'
os.makedirs(save_root, exist_ok=True)
for i in range(len(file_list)):
    file=file_list[i]
    map1=sunpy.map.Map(file)
    with propagate_with_solar_surface():
        rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
    for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                'bunit', 'exptime']:
        if key in map1.meta:
            rot_map1.meta[key] = map1.meta[key]
    rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
    rot_map1.meta.pop('BLANK', None)
    rot_map1.save(os.path.join(save_root, os.path.basename(file_list[i]).replace('.fits','_submap.fits')),overwrite=True)

file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\inclination\*.fits')
save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\inclination_submap'
os.makedirs(save_root, exist_ok=True)
for i in range(len(file_list)):
    file=file_list[i]
    map1=sunpy.map.Map(file)
    with propagate_with_solar_surface():
        rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
    for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                'bunit', 'exptime']:
        if key in map1.meta:
            rot_map1.meta[key] = map1.meta[key]
    rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
    rot_map1.meta.pop('BLANK', None)
    rot_map1.save(os.path.join(save_root, os.path.basename(file_list[i]).replace('.fits','_submap.fits')),overwrite=True)

5. 将不同AIA波段分出来

In [4]:
#这段代码是把一块下好的AIA不同波段分出来
list1=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\C\20120704_090000_UTC\JSOC_20261118_002083\*.fits')
list1.sort()
target_path=r'E:\Learning\PHD3rd\sunspotscar\data\C\20120704_090000_UTC'
# print(list1[0][-19:-16])
for i in range(len(list1)):
    if list1[i][-19:-16]=='131':
        shutil.copy(list1[i],os.path.join(target_path,'131'))
    if list1[i][-19:-16]=='211':
        shutil.copy(list1[i],os.path.join(target_path,'211'))
    if list1[i][-19:-16]=='304':
        shutil.copy(list1[i],os.path.join(target_path,'304'))